In [ ]:
import pandas as pd
import os
import gc

BARE_DIR = '/content/drive/MyDrive/ILGC-V/bare_data/'
CROP_DIR = '/content/drive/MyDrive/ILGC-V/crop_data/'

def count_parquet_folder(folder):
    total = bare = crop = 0
    for fname in sorted(os.listdir(folder)):
        if not fname.endswith('.parquet'):
            continue
        path = os.path.join(folder, fname)
        # Read only the target column — tiny fraction of RAM
        df = pd.read_parquet(path, columns=['target'])
        df = df[df['target'] != 0]
        b = (df['target'] == 2).sum()
        c = (df['target'] == 1).sum()
        print(f"  {fname}: bare={b:,}  crop={c:,}")
        total += len(df)
        bare  += b
        crop  += c
        del df; gc.collect()
    print(f"\n  TOTAL rows : {total:,}")
    print(f"  Bare pixels: {bare:,}  ({100*bare/total:.1f}%)")
    print(f"  Crop pixels: {crop:,}  ({100*crop/total:.1f}%)")
    return total, bare, crop

print("── Bare months ──────────────────────────────────────────")
count_parquet_folder(BARE_DIR)

print("\n── Crop months ──────────────────────────────────────────")
count_parquet_folder(CROP_DIR)

── Bare months ──────────────────────────────────────────
  Copy of Amritsar_S2_may.parquet: bare=30,332,357  crop=9,019,143
  Copy of Patiala_S2_May.parquet: bare=6,951,442  crop=6,100,214
  Copy of ambala_202405_may.parquet: bare=2,404,314  crop=5,239,274
  Copy of bhiwani_202405_may.parquet: bare=6,636,889  crop=2,732,059
  Copy of faridkot_202405_may.parquet: bare=6,074,973  crop=1,258,795
  Copy of firozpur_202405_may.parquet: bare=21,375,925  crop=7,010,305
  Copy of gurdaspur_202405_may.parquet: bare=16,712,725  crop=6,545,485
  Copy of hoshiarpur_202405_may.parquet: bare=4,802,646  crop=9,239,034
  Copy of jalandhar_202405_may.parquet: bare=9,827,492  crop=10,077,184
  Copy of kapurthala_202405_may.parquet: bare=6,156,616  crop=6,337,621
  Copy of ludhiana_202405_may.parquet: bare=14,289,665  crop=9,614,870
  Copy of mansa_202405_may.parquet: bare=12,298,451  crop=1,730,295
  Copy of moga_202405_may.parquet: bare=16,384,801  crop=3,436,606
  Copy of muktsar_202405_may.parquet: 

(256946483, np.int64(277685), np.int64(256668798))

In [ ]:
import pandas as pd
import numpy as np
import os
import gc

BARE_DIR  = '/content/drive/MyDrive/ILGC-V/bare_data/'
CROP_DIR  = '/content/drive/MyDrive/ILGC-V/crop_data/'
BAND_COLS = ['B11','B12','B2','B3','B4','B5','B6','B7','B8','B8A']

# How many pixels per class — keep this reasonable for RAM
# 500k each = 1M total = very safe
# 1M each   = 2M total = safe
# 2M each   = 4M total = should be fine on Colab
N_PER_CLASS = 1_000_000

def sample_from_folder(folder, target_class, n_needed):
    """
    Sample n_needed rows of a specific class from all parquets in a folder.
    Reads only band cols + target — RAM safe.
    """
    collected = []
    collected_n = 0

    files = sorted([f for f in os.listdir(folder) if f.endswith('.parquet')])
    # Spread sample evenly across files
    n_per_file = max(1, n_needed // len(files))

    for fname in files:
        if collected_n >= n_needed:
            break
        path = os.path.join(folder, fname)
        df = pd.read_parquet(path, columns=BAND_COLS + ['target'])
        df = df[df['target'] == target_class]

        take = min(n_per_file, len(df), n_needed - collected_n)
        df   = df.sample(n=take, random_state=42).reset_index(drop=True)
        collected.append(df)
        collected_n += take
        print(f"  {fname}: took {take:,} (class={target_class})")
        del df; gc.collect()

    result = pd.concat(collected, ignore_index=True)
    del collected; gc.collect()
    return result

# ── Sample bare pixels from bare folder ──────────────────────
print(f"Sampling {N_PER_CLASS:,} BARE pixels from bare months...")
df_bare_sample = sample_from_folder(BARE_DIR, target_class=2, n_needed=N_PER_CLASS)

# ── Sample crop pixels from crop folder ──────────────────────
print(f"\nSampling {N_PER_CLASS:,} CROP pixels from crop months...")
df_crop_sample = sample_from_folder(CROP_DIR, target_class=1, n_needed=N_PER_CLASS)

# ── Merge and shuffle ─────────────────────────────────────────
print("\nMerging and shuffling...")
df_combined = pd.concat([df_bare_sample, df_crop_sample], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)
del df_bare_sample, df_crop_sample; gc.collect()

# Binarize target: bare=1, crop=0
df_combined['target'] = (df_combined['target'] == 2).astype(np.int8)

# ── Summary ───────────────────────────────────────────────────
print(f"\n── Combined dataset ─────────────────────────────────────")
print(f"  Total rows : {len(df_combined):,}")
print(f"  Bare (1)   : {(df_combined['target']==1).sum():,}")
print(f"  Crop (0)   : {(df_combined['target']==0).sum():,}")
print(f"\nReady to train ✅")

Sampling 1,000,000 BARE pixels from bare months...
  Copy of Amritsar_S2_may.parquet: took 62,500 (class=2)
  Copy of Patiala_S2_May.parquet: took 62,500 (class=2)
  Copy of ambala_202405_may.parquet: took 62,500 (class=2)
  Copy of bhiwani_202405_may.parquet: took 62,500 (class=2)
  Copy of faridkot_202405_may.parquet: took 62,500 (class=2)
  Copy of firozpur_202405_may.parquet: took 62,500 (class=2)
  Copy of gurdaspur_202405_may.parquet: took 62,500 (class=2)
  Copy of hoshiarpur_202405_may.parquet: took 62,500 (class=2)
  Copy of jalandhar_202405_may.parquet: took 62,500 (class=2)
  Copy of kapurthala_202405_may.parquet: took 62,500 (class=2)
  Copy of ludhiana_202405_may.parquet: took 62,500 (class=2)
  Copy of mansa_202405_may.parquet: took 62,500 (class=2)
  Copy of moga_202405_may.parquet: took 62,500 (class=2)
  Copy of muktsar_202405_may.parquet: took 62,500 (class=2)
  Copy of rupnagar_202405_may.parquet: took 62,500 (class=2)
  Copy of sangrur_202405_may.parquet: took 62,50

In [ ]:
SAVE_PATH = '/content/drive/MyDrive/ILGC-V/combined_training_data.parquet'
df_combined.to_parquet(SAVE_PATH, index=False)
print(f"Saved → {SAVE_PATH}")
print(f"File size: {os.path.getsize(SAVE_PATH)/1e6:.1f} MB")

Saved → /content/drive/MyDrive/ILGC-V/combined_training_data.parquet
File size: 35.4 MB


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import gc
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, classification_report,
                             confusion_matrix, roc_auc_score)

# ── Config ────────────────────────────────────────────────────
DATA_PATH  = '/content/drive/MyDrive/ILGC-V/combined_training_data.parquet'
MODEL_PATH = '/content/drive/MyDrive/ILGC-V/xgb_bare_soil.joblib'
BAND_COLS  = ['B11','B12','B2','B3','B4','B5','B6','B7','B8','B8A']
SEED       = 42


# ── 1. Load data ──────────────────────────────────────────────
print("Loading data...")
df = pd.read_parquet(DATA_PATH)
print(f"  Shape: {df.shape}")
print(f"  Bare (1): {(df['target']==1).sum():,}  Crop (0): {(df['target']==0).sum():,}")


# ── 2. Feature engineering — spectral indices ─────────────────
print("\nComputing spectral indices...")

def add_indices(df):
    B2  = df['B2'].values.astype(np.float32)
    B3  = df['B3'].values.astype(np.float32)
    B4  = df['B4'].values.astype(np.float32)
    B8  = df['B8'].values.astype(np.float32)
    B11 = df['B11'].values.astype(np.float32)
    B12 = df['B12'].values.astype(np.float32)

    eps = 1e-8  # avoid division by zero

    # Bare Soil Index — most important for this task
    df['BSI']       = ((B11 + B4) - (B8 + B2)) / ((B11 + B4) + (B8 + B2) + eps)

    # Vegetation indices — inverse signal of bare soil
    df['NDVI']      = (B8 - B4)  / (B8  + B4  + eps)
    df['NDWI']      = (B3 - B8)  / (B3  + B8  + eps)

    # SWIR ratio — sensitive to soil moisture and bare soil
    df['SWIR_ratio'] = B11 / (B12 + eps)

    # Brightness — bare soil tends to be brighter
    df['brightness'] = (B2 + B3 + B4 + B8) / 4.0

    return df

df = add_indices(df)
gc.collect()

FEATURE_COLS = BAND_COLS + ['BSI', 'NDVI', 'NDWI', 'SWIR_ratio', 'brightness']
print(f"  Features: {len(FEATURE_COLS)}  →  {FEATURE_COLS}")


# ── 3. Train / val split ──────────────────────────────────────
print("\nSplitting train/val (90/10)...")
X = df[FEATURE_COLS].values.astype(np.float32)
y = df['target'].values.astype(np.int8)
del df; gc.collect()

X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=SEED)
del X; gc.collect()

print(f"  Train: {len(y_tr):,}  Val: {len(y_va):,}")


# ── 4. XGBoost — best settings for binary classification ──────
print("\nTraining XGBoost...")

model = xgb.XGBClassifier(
    # Tree structure
    n_estimators      = 500,
    max_depth         = 7,
    min_child_weight  = 5,
    subsample         = 0.8,        # row sampling per tree
    colsample_bytree  = 0.8,        # feature sampling per tree

    # Learning
    learning_rate     = 0.05,
    gamma             = 0.1,        # min loss reduction to split

    # Class balance — bare soil may still be over/under represented
    scale_pos_weight  = (y_tr == 0).sum() / (y_tr == 1).sum(),

    # Speed
    tree_method       = 'hist',     # fast histogram method
    device            = 'cuda',     # change to 'cpu' if no GPU

    # Regularisation
    reg_alpha         = 0.1,        # L1
    reg_lambda        = 1.0,        # L2

    # Eval
    eval_metric       = ['logloss', 'auc'],
    early_stopping_rounds = 20,

    random_state      = SEED,
    verbosity         = 1,
)

model.fit(
    X_tr, y_tr,
    eval_set        = [(X_tr, y_tr), (X_va, y_va)],
    verbose         = 10,    # print every 10 rounds
)

print(f"\nBest iteration: {model.best_iteration}")


# ── 5. Evaluation ─────────────────────────────────────────────
print("\nEvaluating on validation set...")
y_pred      = model.predict(X_va)
y_pred_prob = model.predict_proba(X_va)[:, 1]

print(f"\n── Validation Results ───────────────────────────────────")
print(classification_report(y_va, y_pred,
                             target_names=['Crop (0)', 'Bare (1)'], digits=4))

cm = confusion_matrix(y_va, y_pred)
print(f"Confusion Matrix:")
print(f"               Pred Crop   Pred Bare")
print(f"  True Crop  : {cm[0,0]:>9,}   {cm[0,1]:>9,}")
print(f"  True Bare  : {cm[1,0]:>9,}   {cm[1,1]:>9,}")

print(f"\nROC-AUC: {roc_auc_score(y_va, y_pred_prob):.4f}")
print(f"Bare F1: {f1_score(y_va, y_pred, pos_label=1):.4f}")
print(f"Crop F1: {f1_score(y_va, y_pred, pos_label=0):.4f}")


# ── 6. Feature importance ─────────────────────────────────────
print(f"\n── Feature Importance ───────────────────────────────────")
importance = model.feature_importances_
for feat, imp in sorted(zip(FEATURE_COLS, importance),
                         key=lambda x: x[1], reverse=True):
    bar = '█' * int(imp * 200)
    print(f"  {feat:<15} {imp:.4f}  {bar}")


# ── 7. Save model ─────────────────────────────────────────────
joblib.dump(model, MODEL_PATH)
print(f"\nModel saved → {MODEL_PATH}")

# Verify
loaded = joblib.load(MODEL_PATH)
test_pred = loaded.predict(X_va[:5])
print(f"Load verification — sample predictions: {test_pred}")
print(f"✅ Model saved and verified")

Loading data...
  Shape: (1999990, 11)
  Bare (1): 1,000,000  Crop (0): 999,990

Computing spectral indices...
  Features: 15  →  ['B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'BSI', 'NDVI', 'NDWI', 'SWIR_ratio', 'brightness']

Splitting train/val (90/10)...
  Train: 1,799,991  Val: 199,999

Training XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [05:33:28] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [05:33:28] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()


[0]	validation_0-logloss:0.64443	validation_0-auc:0.99993	validation_1-logloss:0.64443	validation_1-auc:0.99994
[10]	validation_0-logloss:0.33695	validation_0-auc:0.99999	validation_1-logloss:0.33697	validation_1-auc:0.99999
[20]	validation_0-logloss:0.18998	validation_0-auc:0.99999	validation_1-logloss:0.19000	validation_1-auc:0.99999
[30]	validation_0-logloss:0.11084	validation_0-auc:0.99999	validation_1-logloss:0.11086	validation_1-auc:1.00000
[40]	validation_0-logloss:0.06579	validation_0-auc:0.99999	validation_1-logloss:0.06581	validation_1-auc:1.00000
[50]	validation_0-logloss:0.03949	validation_0-auc:1.00000	validation_1-logloss:0.03951	validation_1-auc:1.00000
[60]	validation_0-logloss:0.02388	validation_0-auc:1.00000	validation_1-logloss:0.02389	validation_1-auc:1.00000
[70]	validation_0-logloss:0.01452	validation_0-auc:1.00000	validation_1-logloss:0.01454	validation_1-auc:1.00000
[80]	validation_0-logloss:0.00888	validation_0-auc:1.00000	validation_1-logloss:0.00890	validatio

In [ ]:
# Test on a held-out district parquet that was NOT in training
# e.g. use Amritsar November (which is a different season too)
TEST_PATH = '/content/drive/MyDrive/ILGC-V/parquet/Karnataka/Bangalore_Urban/bangalore_urban_202403_march.parquet'

import pandas as pd
import numpy as np
import joblib
import gc

BAND_COLS = ['B11','B12','B2','B3','B4','B5','B6','B7','B8','B8A']

model = joblib.load('/content/drive/MyDrive/ILGC-V/xgb_bare_soil.joblib')

# Load only annotated pixels for evaluation
df = pd.read_parquet(TEST_PATH, columns=BAND_COLS + ['target'])
df = df[df['target'] != 0].sample(n=200_000, random_state=42)

# Add indices
eps = 1e-8
df['BSI']        = ((df['B11']+df['B4'])-(df['B8']+df['B2'])) / ((df['B11']+df['B4'])+(df['B8']+df['B2'])+eps)
df['NDVI']       = (df['B8']-df['B4'])  / (df['B8'] +df['B4'] +eps)
df['NDWI']       = (df['B3']-df['B8'])  / (df['B3'] +df['B8'] +eps)
df['SWIR_ratio'] = df['B11'] / (df['B12']+eps)
df['brightness'] = (df['B2']+df['B3']+df['B4']+df['B8']) / 4.0

FEATURE_COLS = BAND_COLS + ['BSI','NDVI','NDWI','SWIR_ratio','brightness']
X_test = df[FEATURE_COLS].values.astype(np.float32)
y_test = (df['target'] == 2).values.astype(np.int8)
del df; gc.collect()

from sklearn.metrics import classification_report, roc_auc_score
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print("── Cross-district Test (Amritsar November) ──────────────")
print(classification_report(y_test, y_pred,
      target_names=['Crop (0)', 'Bare (1)'], digits=4))
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_prob):.4f}")

── Cross-district Test (Amritsar November) ──────────────
              precision    recall  f1-score   support

    Crop (0)     1.0000    0.9926    0.9963     96001
    Bare (1)     0.9932    1.0000    0.9966    103999

    accuracy                         0.9965    200000
   macro avg     0.9966    0.9963    0.9965    200000
weighted avg     0.9965    0.9965    0.9965    200000

ROC-AUC: 1.0000


In [ ]:
# Test on a held-out district parquet that was NOT in training
# Bangalore Urban, Karnataka — completely different state, soil type, and climate
TEST_PATH = '/content/drive/MyDrive/ILGC-V/parquet/Karnataka/Bangalore_Urban/bangalore_urban_202403_march.parquet'

import pandas as pd
import numpy as np
import joblib
import gc

BAND_COLS    = ['B11','B12','B2','B3','B4','B5','B6','B7','B8','B8A']
FEATURE_COLS = BAND_COLS + ['BSI','NDVI','NDWI','SWIR_ratio','brightness']
MODEL_PATH   = '/content/drive/MyDrive/ILGC-V/xgb_bare_soil.joblib'

model = joblib.load(MODEL_PATH)
print(f'Model loaded ✅')

# Load only annotated pixels
df = pd.read_parquet(TEST_PATH, columns=BAND_COLS + ['target'])
df = df[df['target'] != 0].sample(n=200_000, random_state=42)
print(f'Loaded {len(df):,} annotated pixels')
print(f'Bare (target=2): {(df["target"]==2).sum():,}')
print(f'Crop (target=1): {(df["target"]==1).sum():,}')

# Compute spectral indices
eps = 1e-8
df['BSI']        = ((df['B11']+df['B4'])-(df['B8']+df['B2'])) / \
                   ((df['B11']+df['B4'])+(df['B8']+df['B2'])+eps)
df['NDVI']       = (df['B8'] - df['B4']) / (df['B8']  + df['B4']  + eps)
df['NDWI']       = (df['B3'] - df['B8']) / (df['B3']  + df['B8']  + eps)
df['SWIR_ratio'] = df['B11'] / (df['B12'] + eps)
df['brightness'] = (df['B2'] + df['B3'] + df['B4'] + df['B8']) / 4.0

X_test = df[FEATURE_COLS].values.astype(np.float32)
y_test = (df['target'] == 2).values.astype(np.int8)   # bare=1, crop=0
del df; gc.collect()

# Predict
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

# Evaluate
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

print("\n── Cross-Region Test (Bangalore Urban, Karnataka, March 2024) ──")
print(classification_report(y_test, y_pred,
      target_names=['Crop (0)', 'Bare (1)'], digits=4))

cm = confusion_matrix(y_test, y_pred)
print(f'Confusion Matrix:')
print(f'               Pred Crop   Pred Bare')
print(f'  True Crop  : {cm[0,0]:>9,}   {cm[0,1]:>9,}')
print(f'  True Bare  : {cm[1,0]:>9,}   {cm[1,1]:>9,}')
print(f'\nROC-AUC: {roc_auc_score(y_test, y_pred_prob):.4f}')

Model loaded ✅
Loaded 200,000 annotated pixels
Bare (target=2): 68,120
Crop (target=1): 131,880

── Cross-Region Test (Bangalore Urban, Karnataka, March 2024) ──
              precision    recall  f1-score   support

    Crop (0)     0.9999    0.9651    0.9822    131880
    Bare (1)     0.9368    0.9998    0.9673     68120

    accuracy                         0.9769    200000
   macro avg     0.9683    0.9825    0.9747    200000
weighted avg     0.9784    0.9769    0.9771    200000

Confusion Matrix:
               Pred Crop   Pred Bare
  True Crop  :   127,282       4,598
  True Bare  :        13      68,107

ROC-AUC: 0.9999


In [ ]:
import pandas as pd
import numpy as np
import joblib
import gc
import os
from glob import glob

# ── CONFIGURATION ─────────────────────────────────────────────
MODEL_PATH    = '/content/drive/MyDrive/ILGC-V/xgb_bare_soil.joblib'
INPUT_FOLDER  = '/content/drive/MyDrive/ILGC-V/parquet_exports/'  # Folder with your Sentinel-2 data
OUTPUT_FOLDER = '/content/drive/MyDrive/ILGC-V/parquet_exports_bare/' # Folder to save results
THRESHOLD     = 0.5  # Probability threshold for bare soil

# Ensure output directory exists
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ── 1. LOAD MODEL ─────────────────────────────────────────────
model = joblib.load(MODEL_PATH)
print(f'Model loaded ✅')

BAND_COLS    = ['B11','B12','B2','B3','B4','B5','B6','B7','B8','B8A']
FEATURE_COLS = BAND_COLS + ['BSI','NDVI','NDWI','SWIR_ratio','brightness']

# ── 2. PROCESS FILES ──────────────────────────────────────────
parquet_files = glob(os.path.join(INPUT_FOLDER, "*.parquet"))

for file_path in parquet_files:
    file_name = os.path.basename(file_path)
    print(f'Processing: {file_name}...')

    # Load the full parquet file (no ground truth needed)
    df = pd.read_parquet(file_path)

    # Compute spectral indices (Exact logic from training)
    eps = 1e-8
    df['BSI']        = ((df['B11']+df['B4'])-(df['B8']+df['B2'])) / \
                       ((df['B11']+df['B4'])+(df['B8']+df['B2'])+eps)
    df['NDVI']       = (df['B8'] - df['B4']) / (df['B8']  + df['B4']  + eps)
    df['NDWI']       = (df['B3'] - df['B8']) / (df['B3']  + df['B8']  + eps)
    df['SWIR_ratio'] = df['B11'] / (df['B12'] + eps)
    df['brightness'] = (df['B2'] + df['B3'] + df['B4'] + df['B8']) / 4.0

    # Prepare features for inference
    X = df[FEATURE_COLS].values.astype(np.float32)

    # Perform predictions
    # We get probabilities first to allow for thresholding
    probs = model.predict_proba(X)[:, 1]
    df['bare_soil_prob'] = probs.astype(np.float32)
    df['prediction']     = (probs >= THRESHOLD).astype(np.int8) # 1=Bare, 0=Non-Bare

    # ── 3. SAVE RESULTS ───────────────────────────────────────
    # Saving to a new file to keep original data clean
    save_path = os.path.join(OUTPUT_FOLDER, f"pred_{file_name}")
    df.to_parquet(save_path, index=False)

    print(f'  Saved → {save_path}')

    # Cleanup memory
    del df, X, probs; gc.collect()

print("\n✅ All files processed successfully.")

Model loaded ✅
Processing: SOC_Poly_1_2025_01.parquet...
  Saved → /content/drive/MyDrive/ILGC-V/parquet_exports_bare/pred_SOC_Poly_1_2025_01.parquet
Processing: SOC_Poly_1_2025_02.parquet...
  Saved → /content/drive/MyDrive/ILGC-V/parquet_exports_bare/pred_SOC_Poly_1_2025_02.parquet
Processing: SOC_Poly_1_2025_04.parquet...
  Saved → /content/drive/MyDrive/ILGC-V/parquet_exports_bare/pred_SOC_Poly_1_2025_04.parquet
Processing: SOC_Poly_1_2025_03.parquet...
  Saved → /content/drive/MyDrive/ILGC-V/parquet_exports_bare/pred_SOC_Poly_1_2025_03.parquet
Processing: SOC_Poly_1_2025_06.parquet...
  Saved → /content/drive/MyDrive/ILGC-V/parquet_exports_bare/pred_SOC_Poly_1_2025_06.parquet
Processing: SOC_Poly_1_2025_05.parquet...
  Saved → /content/drive/MyDrive/ILGC-V/parquet_exports_bare/pred_SOC_Poly_1_2025_05.parquet
Processing: SOC_Poly_1_2025_07.parquet...
  Saved → /content/drive/MyDrive/ILGC-V/parquet_exports_bare/pred_SOC_Poly_1_2025_07.parquet
Processing: SOC_Poly_1_2025_08.parquet..

In [ ]:
pd.read_parquet('/content/drive/MyDrive/ILGC-V/parquet_exports_bare/pred_SOC_Poly_10_2025_01.parquet')

,B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,...,poly_name,year,month,BSI,NDVI,NDWI,SWIR_ratio,brightness,bare_soil_prob,prediction
0,1205.0,1232.0,1360.0,1218.0,1485.0,2656.0,3150.0,2981.0,3207.0,3020.0,...,10,2025,1,-0.191459,0.419862,-0.373416,1.582449,1697.75,0.000204,0
1,1205.0,1130.0,1220.0,1035.0,1485.0,2656.0,3150.0,3572.0,3207.0,3020.0,...,10,2025,1,-0.274600,0.550684,-0.490818,1.582449,1739.25,0.000204,0
2,1205.0,1132.0,1242.0,1045.0,1447.0,2843.0,3383.0,3464.0,3424.0,3020.0,...,10,2025,1,-0.282187,0.536483,-0.472163,1.849879,1720.75,0.000204,0
3,1205.0,1140.0,1235.0,1050.0,1447.0,2843.0,3383.0,3482.0,3424.0,3020.0,...,10,2025,1,-0.283889,0.536628,-0.476362,1.849879,1726.75,0.000204,0
4,1174.0,1257.0,1325.0,1182.0,1574.0,2291.0,2509.0,2872.0,2653.0,2435.0,...,10,2025,1,-0.148540,0.416872,-0.368597,1.453210,1659.00,0.000204,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,1198.0,1192.0,1273.0,1065.0,1463.0,2807.0,3365.0,3402.0,3427.0,3269.0,...,10,2025,1,-0.276111,0.523170,-0.455401,1.933501,1733.00,0.000204,0
57,1198.0,1200.0,1282.0,1069.0,1463.0,2807.0,3365.0,3403.0,3427.0,3269.0,...,10,2025,1,-0.276307,0.521914,-0.452721,1.933501,1738.50,0.000204,0
58,1198.0,1156.0,1280.0,1072.0,1479.0,2698.0,3192.0,3403.0,3301.0,3269.0,...,10,2025,1,-0.253161,0.520894,-0.453342,1.757479,1727.75,0.000204,0
59,1198.0,1189.0,1313.0,1101.0,1479.0,2698.0,3192.0,3273.0,3301.0,3269.0,...,10,2025,1,-0.238069,0.496571,-0.427388,1.757479,1719.00,0.000204,0


In [ ]:
import pandas as pd
import numpy as np
import joblib
import gc
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

BAND_COLS    = ['B11','B12','B2','B3','B4','B5','B6','B7','B8','B8A']
FEATURE_COLS = BAND_COLS + ['BSI','NDVI','NDWI','SWIR_ratio','brightness']
MODEL_PATH   = '/content/drive/MyDrive/ILGC-V/xgb_bare_soil.joblib'

# ── Add as many unseen districts as you have ─────────────────
# These should be districts NEVER seen during training
TEST_FILES = {
    'Bangalore Urban (Karnataka, March)' : '/content/drive/MyDrive/ILGC-V/parquet/Karnataka/Bangalore_Urban/bangalore_urban_202403_march.parquet'    # Add more here ↓
    # 'Mysuru (Karnataka, May)'          : '/content/drive/MyDrive/ILGC-V/parquet/Karnataka/Mysuru/mysuru_202405_may.parquet',
    # 'Jaipur (Rajasthan, May)'          : '/content/drive/MyDrive/ILGC-V/parquet/Rajasthan/Jaipur/jaipur_202405_may.parquet',
}

model = joblib.load(MODEL_PATH)
print(f'Model loaded ✅\n')

def evaluate(path, label):
    print(f'── {label} ──────────────────────────────────────────')

    # Load ALL annotated pixels — no sampling
    df = pd.read_parquet(path, columns=BAND_COLS + ['target'])
    df = df[df['target'] != 0].reset_index(drop=True)

    n_bare = (df['target'] == 2).sum()
    n_crop = (df['target'] == 1).sum()
    print(f'  Total annotated: {len(df):,}  (bare={n_bare:,}  crop={n_crop:,})')

    if len(df) == 0:
        print('  No annotated pixels — skipping.')
        return

    # Spectral indices
    eps = 1e-8
    df['BSI']        = ((df['B11']+df['B4'])-(df['B8']+df['B2'])) / \
                       ((df['B11']+df['B4'])+(df['B8']+df['B2'])+eps)
    df['NDVI']       = (df['B8'] - df['B4'])  / (df['B8']  + df['B4']  + eps)
    df['NDWI']       = (df['B3'] - df['B8'])  / (df['B3']  + df['B8']  + eps)
    df['SWIR_ratio'] = df['B11'] / (df['B12'] + eps)
    df['brightness'] = (df['B2'] + df['B3'] + df['B4'] + df['B8']) / 4.0

    X = df[FEATURE_COLS].values.astype(np.float32)
    y = (df['target'] == 2).values.astype(np.int8)
    del df; gc.collect()

    # Predict
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    # Report
    print(classification_report(y, y_pred,
          target_names=['Crop (0)', 'Bare (1)'], digits=4))

    cm = confusion_matrix(y, y_pred)
    print(f'  Confusion Matrix:')
    print(f'                 Pred Crop   Pred Bare')
    print(f'    True Crop  : {cm[0,0]:>9,}   {cm[0,1]:>9,}')
    print(f'    True Bare  : {cm[1,0]:>9,}   {cm[1,1]:>9,}')
    print(f'  ROC-AUC      : {roc_auc_score(y, y_prob):.4f}')
    del X, y, y_pred, y_prob; gc.collect()
    print()

for label, path in TEST_FILES.items():
    evaluate(path, label)

Model loaded ✅

── Bangalore Urban (Karnataka, March) ──────────────────────────────────────────
  Total annotated: 3,641,912  (bare=1,246,162  crop=2,395,750)
              precision    recall  f1-score   support

    Crop (0)     0.9999    0.9651    0.9822   2395750
    Bare (1)     0.9371    0.9999    0.9675   1246162

    accuracy                         0.9770   3641912
   macro avg     0.9685    0.9825    0.9748   3641912
weighted avg     0.9784    0.9770    0.9772   3641912

  Confusion Matrix:
                 Pred Crop   Pred Bare
    True Crop  : 2,312,138      83,612
    True Bare  :       162   1,246,000
  ROC-AUC      : 0.9999



In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import joblib
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
import gc
import psutil
import os

def ram():
    return psutil.virtual_memory().available / 1e9

# ══════════════════════════════════════════════════════════════
#  CONFIG  — edit these paths before running
# ══════════════════════════════════════════════════════════════
MODEL_PATH  = '/content/drive/MyDrive/ILGC-V/xgb_bare_soil.joblib'
IN_PARQUET  = '/content/drive/MyDrive/ILGC-V/parquet/Karnataka/Bangalore_Urban/bangalore_urban_202403_march.parquet'
OUTPUT_TIFF = '/content/drive/MyDrive/ILGC-V/bangalore_bare_soil_output.tif'
BAND_COLS   = ['B11','B12','B2','B3','B4','B5','B6','B7','B8','B8A']
FEAT_COLS   = BAND_COLS + ['BSI','NDVI','NDWI','SWIR_ratio','brightness']
CHUNK_ROWS  = 500_000   # rows per batch — lower if RAM is tight
THRESHOLD   = 0.5       # bare-soil probability cutoff
# ══════════════════════════════════════════════════════════════

# ── 1. Load model ─────────────────────────────────────────────
model = joblib.load(MODEL_PATH)
print(f'✅ Model loaded | RAM free: {ram():.1f} GB')

# ── 2. Pass 1 — build the raster grid from lon/lat ────────────
print(f'\nPass 1: reading coordinates... | RAM free: {ram():.1f} GB')
coords = pd.read_parquet(IN_PARQUET, columns=['longitude', 'latitude'])
lons = np.round(coords['longitude'].values, 6).astype(np.float32)
lats = np.round(coords['latitude'].values,  6).astype(np.float32)
del coords; gc.collect()

unique_lons = np.sort(np.unique(lons))
unique_lats = np.sort(np.unique(lats))[::-1]          # top row first
height, width = len(unique_lats), len(unique_lons)
total_pixels  = len(lons)
print(f'Grid : {height} rows × {width} cols | {total_pixels:,} pixels | RAM free: {ram():.1f} GB')

lon2col  = {v: i for i, v in enumerate(unique_lons)}
lat2row  = {v: i for i, v in enumerate(unique_lats)}
all_cols = np.array([lon2col[v] for v in lons], dtype=np.int32)
all_rows = np.array([lat2row[v] for v in lats], dtype=np.int32)
del lons, lats, lon2col, lat2row; gc.collect()
print(f'Row/col index ready | RAM free: {ram():.1f} GB')

# ── 3. Allocate output rasters ────────────────────────────────
#   Band 1 → binary prediction  (0 = Crop, 1 = Bare Soil)
#   Band 2 → bare-soil probability  (0.0 – 1.0)
raster_pred = np.zeros((height, width), dtype=np.uint8)
raster_prob = np.zeros((height, width), dtype=np.float32)
print(f'Rasters allocated ({height}×{width}) | RAM free: {ram():.1f} GB')

# ── 4. Pass 2 — chunked inference ────────────────────────────
print(f'\nPass 2: inference in chunks of {CHUNK_ROWS:,}...')
pf     = pq.ParquetFile(IN_PARQUET)
cursor = 0

for batch in pf.iter_batches(batch_size=CHUNK_ROWS, columns=BAND_COLS):

    # — build DataFrame from Arrow batch
    chunk = pd.DataFrame(
        {b: np.asarray(batch.column(b), dtype=np.float32) for b in BAND_COLS}
    )
    del batch; gc.collect()

    # — spectral indices (must match training)
    eps = 1e-8
    chunk['BSI']        = ((chunk['B11']+chunk['B4'])-(chunk['B8']+chunk['B2'])) \
                        / ((chunk['B11']+chunk['B4'])+(chunk['B8']+chunk['B2'])+eps)
    chunk['NDVI']       = (chunk['B8']-chunk['B4'])  / (chunk['B8'] +chunk['B4'] +eps)
    chunk['NDWI']       = (chunk['B3']-chunk['B8'])  / (chunk['B3'] +chunk['B8'] +eps)
    chunk['SWIR_ratio'] = chunk['B11'] / (chunk['B12']+eps)
    chunk['brightness'] = (chunk['B2']+chunk['B3']+chunk['B4']+chunk['B8']) / 4.0

    X = chunk[FEAT_COLS].values.astype(np.float32)
    del chunk; gc.collect()

    # — XGBoost predict
    probs = model.predict_proba(X)[:, 1].astype(np.float32)
    pred  = (probs >= THRESHOLD).astype(np.uint8)   # 1 = bare soil
    del X; gc.collect()

    # — write into rasters
    n = len(probs)
    r = all_rows[cursor:cursor + n]
    c = all_cols[cursor:cursor + n]
    raster_pred[r, c] = pred
    raster_prob[r, c] = probs
    del probs, pred, r, c; gc.collect()

    cursor += n
    print(f'  {cursor:>12,} / {total_pixels:,}  '
          f'({100*cursor/total_pixels:.1f}%)  | RAM free: {ram():.1f} GB')

del all_rows, all_cols; gc.collect()

# ── 5. Quick summary ──────────────────────────────────────────
n_bare = int((raster_pred == 1).sum())
n_crop = int((raster_pred == 0).sum())
print(f'\n── Inference results (threshold = {THRESHOLD}) ────────────')
print(f'  Bare soil : {n_bare:>10,}  ({100*n_bare/total_pixels:.1f}%)')
print(f'  Crop      : {n_crop:>10,}  ({100*n_crop/total_pixels:.1f}%)')

# ── 6. Build geo-transform ────────────────────────────────────
min_lon, max_lon = float(unique_lons[0]),  float(unique_lons[-1])
min_lat, max_lat = float(unique_lats[-1]), float(unique_lats[0])
pw = (max_lon - min_lon) / max(width  - 1, 1)   # pixel width
ph = (max_lat - min_lat) / max(height - 1, 1)   # pixel height

transform = from_bounds(
    min_lon - pw/2, min_lat - ph/2,
    max_lon + pw/2, max_lat + ph/2,
    width, height
)

# ── 7. Write GeoTIFF (2 bands, LZW compressed) ───────────────
print(f'\nSaving GeoTIFF → {OUTPUT_TIFF}')
os.makedirs(os.path.dirname(OUTPUT_TIFF), exist_ok=True)

with rasterio.open(
    OUTPUT_TIFF, 'w',
    driver='GTiff',
    height=height, width=width, count=2,
    dtype='float32',
    crs=CRS.from_epsg(4326),
    transform=transform,
    compress='lzw',
    nodata=-1,
) as dst:
    dst.write(raster_pred.astype(np.float32), 1)
    dst.update_tags(1, description='0=Crop  1=Bare_Soil  (threshold=0.5)')
    dst.write(raster_prob, 2)
    dst.update_tags(2, description='Bare soil probability 0.0–1.0')

del raster_pred, raster_prob; gc.collect()

size_mb = os.path.getsize(OUTPUT_TIFF) / 1e6
print(f'✅ Saved  ({size_mb:.1f} MB)')
print(f'   Band 1 → binary prediction   (0=Crop, 1=Bare Soil, threshold={THRESHOLD})')
print(f'   Band 2 → bare-soil probability  (0.0 – 1.0)')
print(f'   CRS    → EPSG:4326  |  Size: {height}×{width}  |  LZW compressed')
print(f'\n── GEE visualisation snippet ─────────────────────────────')
print(f"  var img = ee.Image.loadGeoTIFF('gs://your-bucket/bare_soil_output.tif');")
print(f"  // Band 1 at fixed threshold")
print(f"  Map.addLayer(img.select('b1'), {{min:0,max:1,palette:['green','brown']}}, 'Bare Soil 0.5');")
print(f"  // Band 2 with custom threshold")
print(f"  var thr = 0.7;")
print(f"  Map.addLayer(img.select('b2').gt(thr), {{min:0,max:1,palette:['green','brown']}}, 'Bare Soil '+thr);")

✅ Model loaded | RAM free: 10.6 GB

Pass 1: reading coordinates... | RAM free: 10.6 GB
Grid : 6402 rows × 5604 cols | 22,511,070 pixels | RAM free: 10.3 GB
Row/col index ready | RAM free: 10.3 GB
Rasters allocated (6402×5604) | RAM free: 10.3 GB

Pass 2: inference in chunks of 500,000...
       500,000 / 22,511,070  (2.2%)  | RAM free: 10.2 GB
     1,000,000 / 22,511,070  (4.4%)  | RAM free: 10.3 GB
     1,500,000 / 22,511,070  (6.7%)  | RAM free: 10.3 GB
     2,000,000 / 22,511,070  (8.9%)  | RAM free: 10.3 GB
     2,500,000 / 22,511,070  (11.1%)  | RAM free: 10.3 GB
     3,000,000 / 22,511,070  (13.3%)  | RAM free: 10.3 GB
     3,500,000 / 22,511,070  (15.5%)  | RAM free: 10.3 GB
     4,000,000 / 22,511,070  (17.8%)  | RAM free: 10.3 GB
     4,500,000 / 22,511,070  (20.0%)  | RAM free: 10.3 GB
     5,000,000 / 22,511,070  (22.2%)  | RAM free: 10.3 GB
     5,500,000 / 22,511,070  (24.4%)  | RAM free: 10.3 GB
     6,000,000 / 22,511,070  (26.7%)  | RAM free: 10.3 GB
     6,500,000 / 22